# Hybrid FL-Pooling ViT Framework for Early Cerebral Palsy Detection via Infant Motor Pattern Analysis

**Pipeline**
1. Data: infant videos / keypoint files (`SoftwareImpacts/SIMPAC-2022-116`) -> frame extraction, resize, normalise, denoise, pose estimation, temporal alignment
2. Augmentation: rotation, translation, scale, temporal change (+ occlusion simulation, class balancing)
3. Model: **Hybrid PVT** = convolutional stem + skeleton kinematic branch + Pooling-based Vision Transformer (pooled attention + token pooling)
4. **Federated learning** (FedAvg / optional FedProx, optional DP clip+noise) across simulated institutions
5. Evaluation: Accuracy, Sensitivity, Specificity, Precision, F1, AUC-ROC, inference latency, communication rounds / MB, leave-one-institution-out
6. Ablation: Full (PVT+Pooling+FL), w/o Pooling, w/o FL, w/o Transformer
7. Baselines: GCN-GRU, DCNN, Bi-GRU-LSTM, TransCP-Net

> **Data note.** The notebook tries to clone the GitHub dataset and auto-discover videos / keypoint files. If nothing usable is found (or there is no internet) it falls back to a **synthetic infant-skeleton generator** so the whole pipeline still runs end-to-end. Synthetic numbers are only a pipeline check, never clinical results. Section 2 prints exactly which source is used.

In [ ]:
import os, re, sys, glob, json, math, time, random, copy, warnings, subprocess
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve)
warnings.filterwarnings("ignore")

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)

# ------------------------------- CONFIG -------------------------------
QUICK = True           # True: small/fast run. False: full-length training
T, J = 32, 17          # aligned frames per clip, joints (COCO-17 layout)
WIN, STRIDE = 96, 48   # raw-frame window / stride used to cut long videos
N_INST = 5             # number of (simulated) institutions / FL clients
DIRICHLET_ALPHA = 0.5  # label-skew when institutions are not given in the data
BATCH, LR = 16, 2e-3
ROUNDS       = 12 if QUICK else 40
LOCAL_EPOCHS = 2  if QUICK else 3
CENTRAL_EPOCHS = ROUNDS * LOCAL_EPOCHS      # same number of passes over data
LOIO_ROUNDS  = 8  if QUICK else 25
FEDPROX_MU   = 0.0     # >0 enables FedProx
DP_CLIP, DP_SIGMA = 0.0, 0.0   # >0 enables update clipping / Gaussian noise (DP-style)
PATIENCE = 6
BASELINE_MODE = "central"      # "central" or "fed" for GCN-GRU / DCNN / Bi-GRU-LSTM / TransCP-Net
OUT = "outputs"; os.makedirs(OUT, exist_ok=True)

## 1. Skeleton utilities, pose estimation and preprocessing
Keypoints follow the COCO-17 order. Mappings from MediaPipe (33), OpenPose BODY_25 and OpenPose COCO-18 are provided.

In [ ]:
COCO17 = ["nose","l_eye","r_eye","l_ear","r_ear","l_sho","r_sho","l_elb","r_elb",
          "l_wri","r_wri","l_hip","r_hip","l_kne","r_kne","l_ank","r_ank"]
PARENT = [0,0,0,1,2,0,0,5,6,7,8,5,6,11,12,13,14]
LR_PAIRS = [(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]
MP_TO_COCO = [0,2,5,7,8,11,12,13,14,15,16,23,24,25,26,27,28]
B25_TO_COCO = [0,16,15,18,17,5,2,6,3,7,4,12,9,13,10,14,11]
B18_TO_COCO = [0,15,14,17,16,5,2,6,3,7,4,11,8,12,9,13,10]

def to_coco17(a):
    n = a.shape[1]
    if n == 17: return a
    if n == 33: return a[:, MP_TO_COCO]
    if n == 25: return a[:, B25_TO_COCO]
    if n == 18: return a[:, B18_TO_COCO]
    raise ValueError(f"unsupported joint count {n}")

def fill_missing(a):
    # a: (T,J,2) with NaN for occluded / undetected joints -> linear interpolation over time
    a = a.copy(); Tn = a.shape[0]; t = np.arange(Tn)
    for j in range(a.shape[1]):
        for c in range(2):
            x = a[:, j, c]; m = np.isfinite(x)
            if m.all(): continue
            if m.sum() == 0: continue
            a[:, j, c] = np.interp(t, t[m], x[m])
    if np.isnan(a).any():           # joints never seen: use mean of visible joints in that frame
        fm = np.nanmean(a, axis=1, keepdims=True)
        a = np.where(np.isnan(a), np.broadcast_to(fm, a.shape), a)
        a = np.nan_to_num(a)
    return a

def smooth(a, k=5):
    pad = k // 2
    ap = np.pad(a, ((pad, pad), (0, 0), (0, 0)), mode="edge")
    ker = np.ones(k) / k
    out = np.empty_like(a)
    for j in range(a.shape[1]):
        for c in range(2):
            out[:, j, c] = np.convolve(ap[:, j, c], ker, mode="valid")
    return out

def temporal_align(a, Tn=T):
    src = np.linspace(0, 1, a.shape[0]); dst = np.linspace(0, 1, Tn)
    out = np.empty((Tn, a.shape[1], 2), dtype=np.float64)
    for j in range(a.shape[1]):
        for c in range(2):
            out[:, j, c] = np.interp(dst, src, a[:, j, c])
    return out

def normalise_pose(p):
    # torso-centred and torso-length scaled (removes camera position / zoom / infant size)
    center = p[:, [5, 6, 11, 12]].mean(1, keepdims=True)
    p = p - center
    tl = np.linalg.norm(p[:, [5, 6]].mean(1) - p[:, [11, 12]].mean(1), axis=-1).mean()
    return p / max(tl, 1e-6)

def preprocess_clip(raw):
    # raw: (T0,17,2) pixel/normalised coords, NaN or (0,0) = missing
    raw = raw.astype(np.float64).copy()
    raw[(raw[..., 0] == 0) & (raw[..., 1] == 0)] = np.nan
    p = fill_missing(raw)
    p = smooth(p, 5)
    p = temporal_align(p, T)
    return normalise_pose(p).astype(np.float32)

def make_windows(raw):
    out = []
    if len(raw) < 16: return out
    if len(raw) <= WIN:
        out.append(preprocess_clip(raw))
    else:
        for s in range(0, len(raw) - WIN + 1, STRIDE):
            out.append(preprocess_clip(raw[s:s + WIN]))
    return out

# ---------------- video -> keypoints (frame extraction, resize, normalise, denoise, pose) ----------------
def video_to_keypoints(path, max_frames=900, size=256):
    import cv2, mediapipe as mp
    pose = mp.solutions.pose.Pose(static_image_mode=False, model_complexity=1,
                                  min_detection_confidence=0.3, min_tracking_confidence=0.3)
    cap = cv2.VideoCapture(path); kps = []
    while len(kps) < max_frames:
        ok, fr = cap.read()
        if not ok: break
        fr = cv2.resize(fr, (size, size))                                  # resize
        fr = cv2.bilateralFilter(fr, 5, 40, 40)                            # denoise (edge preserving)
        fr = cv2.normalize(fr, None, 0, 255, cv2.NORM_MINMAX)              # intensity normalisation
        res = pose.process(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))            # pose estimation
        if res.pose_landmarks:
            kps.append([[l.x, l.y] if l.visibility > 0.2 else [np.nan, np.nan]
                        for l in res.pose_landmarks.landmark])
        else:
            kps.append([[np.nan, np.nan]] * 33)
    cap.release(); pose.close()
    return np.array(kps, dtype=np.float64)

## 2. Dataset discovery / loading (with synthetic fallback)
`label_from_path` decides CP vs. typical from folder / file names (deepest component first). **Check the printed label counts and edit that function if the repository uses different naming.**

In [ ]:
REPO_URL, DATA_ROOT = "https://github.com/SoftwareImpacts/SIMPAC-2022-116", "SIMPAC-2022-116"
VIDEO_EXT = (".mp4", ".avi", ".mov", ".mkv", ".mpg", ".mpeg")
KP_EXT = (".npy", ".npz", ".csv")
POS_RE = re.compile(r"(^|[^a-z])(cp|cerebral|abnormal|atypical|pathologic\w*|high[_-]?risk|absent|poor|pr|cr)([^a-z]|$)")
NEG_RE = re.compile(r"(^|[^a-z])(normal|typical|healthy|control\w*|tdc|td|fidgety|low[_-]?risk|nr|fm\+?)([^a-z]|$)")

def label_from_path(p):
    parts = os.path.normpath(p).replace("\\", "/").lower().split("/")
    parts[-1] = os.path.splitext(parts[-1])[0]
    for comp in reversed(parts):             # deepest component first
        if POS_RE.search(comp): return 1
        if NEG_RE.search(comp): return 0
    return None

if not os.path.exists(DATA_ROOT):
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, DATA_ROOT], check=True, timeout=600)
    except Exception as e:
        print("Could not clone dataset:", e)

def load_raw(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in VIDEO_EXT: return video_to_keypoints(path)
    if ext == ".npy": a = np.load(path, allow_pickle=False)
    elif ext == ".npz": z = np.load(path); a = z[z.files[0]]
    else: a = pd.read_csv(path).select_dtypes("number").values
    a = np.asarray(a, dtype=np.float64)
    if a.ndim == 2:
        k_found = None
        for k in (3, 2):
            if a.shape[1] % k == 0 and a.shape[1] // k in (17, 18, 25, 33): k_found = k; break
        if k_found is None: raise ValueError("cannot infer keypoint layout")
        a = a.reshape(a.shape[0], -1, k_found)
    if a.ndim != 3: raise ValueError("bad ndim")
    if a.shape[-1] >= 3:
        xy = a[..., :2].copy(); xy[a[..., 2] < 0.05] = np.nan; a = xy
    return to_coco17(a[..., :2])

def load_real_dataset(root):
    files = [os.path.join(dp, f) for dp, _, fs in os.walk(root) for f in fs
             if os.path.splitext(f)[1].lower() in VIDEO_EXT + KP_EXT]
    print(f"candidate files: {len(files)}")
    P, y, vid = [], [], []
    skipped = {"no_label": 0, "load_error": 0, "too_short": 0}
    for vi, f in enumerate(sorted(files)):
        lab = label_from_path(f)
        if lab is None: skipped["no_label"] += 1; continue
        try: raw = load_raw(f)
        except Exception as e: skipped["load_error"] += 1; continue
        ws = make_windows(raw)
        if not ws: skipped["too_short"] += 1; continue
        for w in ws: P.append(w); y.append(lab); vid.append(vi)
    print("skipped:", skipped)
    if len(P) == 0: return None
    return dict(P=np.stack(P), y=np.array(y), vid=np.array(vid))

# ------------------------- synthetic infant-skeleton generator -------------------------
REST = np.array([[0,-1.3],[-.15,-1.4],[.15,-1.4],[-.3,-1.3],[.3,-1.3],[-.45,-.5],[.45,-.5],
                 [-.8,-.2],[.8,-.2],[-1.0,-.7],[1.0,-.7],[-.3,.5],[.3,.5],[-.45,1.2],[.45,1.2],
                 [-.5,1.9],[.5,1.9]], dtype=np.float64)
LEFT = {7, 9, 13, 15}; LIMB = [7, 8, 9, 10, 13, 14, 15, 16]; DISTAL = {9, 10, 15, 16}

def synth_video(label, ip, rng, T0=WIN, fps=30):
    t = np.arange(T0) / fps
    P = np.repeat(REST[None], T0, 0).copy()
    mild = (label == 1) and rng.random() < 0.4          # mild CP looks close to typical movement
    quiet = (label == 0) and rng.random() < 0.25        # quiet typical infants overlap with CP
    ncomp = 3 if (label == 0 or mild) else int(rng.integers(1, 3))
    amp = rng.uniform(0.04, 0.12) if quiet else (rng.uniform(0.06, 0.17) if mild else (rng.uniform(0.07, 0.20) if label == 0 else rng.uniform(0.02, 0.12)))
    sync = rng.uniform(0.0, 0.3) if (label == 0 or mild) else rng.uniform(0.3, 0.9)   # CP: stiff, synchronised
    shared = rng.uniform(0, 2*np.pi, (ncomp, 2))
    damp = np.ones(2)
    if label == 1 and rng.random() < 0.5: damp[int(rng.integers(0, 2))] = rng.uniform(0.2, 0.6)  # asymmetry
    tremor = (label == 1) and (not mild) and rng.random() < 0.4
    off = {}
    for j in LIMB:
        side = 0 if j in LEFT else 1
        o = np.zeros((T0, 2))
        for k in range(ncomp):
            f = rng.uniform(0.4, 2.8 if label == 0 else 1.8)
            ph = sync * shared[k] + (1 - sync) * rng.uniform(0, 2*np.pi, 2)
            a = amp * rng.uniform(0.5, 1.2) * damp[side]
            o += a * np.stack([np.sin(2*np.pi*f*t + ph[0]), np.sin(2*np.pi*f*t + ph[1])], -1)
        if tremor: o += 0.015 * np.sin(2*np.pi*rng.uniform(5, 8)*t + rng.uniform(0, 6.28))[:, None]
        off[j] = o
    for j in LIMB:
        P[:, j] += off[j] + (off[PARENT[j]] if j in DISTAL else 0)
    P[:, :5] += 0.03 * np.sin(2*np.pi*rng.uniform(0.2, 0.8)*t + rng.uniform(0, 6.28))[:, None, None]
    ang = ip["angle"] + rng.normal(0, 0.1); s = ip["scale"] * rng.uniform(0.9, 1.1)
    R = np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]])
    P = (P @ R.T) * s + np.array([300., 250.]) + rng.normal(0, ip["noise"] * s, P.shape)
    if rng.random() < ip["occl"]:                       # occlusion bursts
        for _ in range(int(rng.integers(1, 4))):
            j = int(rng.integers(0, J)); st = int(rng.integers(0, T0 - 30)); P[st:st + int(rng.integers(8, 30)), j] = np.nan
    return P

def make_synthetic(n_videos=400, n_inst=N_INST, seed=SEED):
    rng = np.random.default_rng(seed)
    inst_params = [dict(angle=rng.uniform(-.4, .4), scale=rng.uniform(.7, 1.4)*80,
                        noise=rng.uniform(.005, .02), occl=rng.uniform(.2, .8),
                        p_cp=rng.uniform(.25, .55)) for _ in range(n_inst)]
    P, y, vid, inst = [], [], [], []
    for v in range(n_videos):
        k = int(rng.integers(0, n_inst)); lab = int(rng.random() < inst_params[k]["p_cp"])
        w = make_windows(synth_video(lab, inst_params[k], rng))
        for ww in w: P.append(ww); y.append(lab); vid.append(v); inst.append(k)
    return dict(P=np.stack(P), y=np.array(y), vid=np.array(vid), inst=np.array(inst))

def partition_dirichlet(vlabels, K, alpha, rng, min_size=4):
    # assigns each *video* to one of K pseudo-institutions with label skew
    for _ in range(200):
        assign = np.zeros(len(vlabels), int)
        for c in np.unique(vlabels):
            idx = np.where(vlabels == c)[0]; rng.shuffle(idx)
            p = rng.dirichlet([alpha] * K); cuts = (np.cumsum(p) * len(idx)).astype(int)[:-1]
            for k, part in enumerate(np.split(idx, cuts)): assign[part] = k
        if min(np.bincount(assign, minlength=K)) >= min_size: return assign
    return np.arange(len(vlabels)) % K

W = load_real_dataset(DATA_ROOT) if os.path.exists(DATA_ROOT) else None
if W is not None and len(np.unique(W["y"])) == 2 and len(np.unique(W["vid"])) >= 20:
    DATA_SOURCE = "REAL (SIMPAC-2022-116)"
    uv = np.unique(W["vid"]); vl = np.array([W["y"][W["vid"] == v][0] for v in uv])
    va = partition_dirichlet(vl, N_INST, DIRICHLET_ALPHA, np.random.default_rng(SEED))
    m = dict(zip(uv, va)); W["inst"] = np.array([m[v] for v in W["vid"]])
    print("NOTE: institution ids are pseudo-institutions (Dirichlet label-skew); replace with real site ids if available.")
else:
    DATA_SOURCE = "SYNTHETIC (fallback - pipeline check only)"
    print("Real data not usable (missing / no labels inferred / <20 videos) -> using synthetic generator.")
    W = make_synthetic()
print("DATA SOURCE:", DATA_SOURCE)
print("windows:", W["P"].shape, "| videos:", len(np.unique(W["vid"])),
      "| CP fraction:", W["y"].mean().round(3), "| windows per institution:", np.bincount(W["inst"]))

## 3. Features, kinematic descriptors, augmentation, datasets
* Conv input tensor `(6, T, J)`: x, y, vx, vy and bone vector (bx, by)
* Skeleton kinematic descriptors: per-joint speed mean/std, acceleration, jerk, left-right synchrony, spectral entropy (movement variability/complexity cues relevant to CP)
* Augmentation: temporal speed/shift change, rotation, scale, translation, noise, occlusion
* Class imbalance handled with a balanced sampler (minority clips drawn more often, each augmented differently)

In [ ]:
def make_features(p):
    v = np.gradient(p, axis=0)
    bone = p - p[:, PARENT]
    f = np.concatenate([p, v, bone], axis=-1)            # (T,J,6)
    return np.ascontiguousarray(f.transpose(2, 0, 1)).astype(np.float32)   # (6,T,J)

def kinematic_descriptors(p):
    v = np.gradient(p, axis=0); a = np.gradient(v, axis=0); jk = np.gradient(a, axis=0)
    sp, ac, jr = (np.linalg.norm(z, axis=-1) for z in (v, a, jk))       # (T,J)
    f = [sp.mean(0), sp.std(0), ac.mean(0), jr.mean(0)]
    sync = []
    for l, r in LR_PAIRS:
        if sp[:, l].std() < 1e-8 or sp[:, r].std() < 1e-8: sync.append(0.0)
        else: sync.append(float(np.corrcoef(sp[:, l], sp[:, r])[0, 1]))
    e = sp.mean(1); ps = np.abs(np.fft.rfft(e - e.mean())) ** 2 + 1e-12; ps = ps / ps.sum()
    ent = float(-(ps * np.log(ps)).sum() / np.log(len(ps)))
    return np.concatenate(f + [np.array(sync), np.array([ent])]).astype(np.float32)

DESC_DIM = len(kinematic_descriptors(W["P"][0]))
print("conv input:", make_features(W["P"][0]).shape, "| descriptor dim:", DESC_DIM)

def interp_time(p, grid):
    i0 = np.floor(grid).astype(int); i1 = np.minimum(i0 + 1, p.shape[0] - 1)
    w = (grid - i0)[:, None, None]
    return p[i0] * (1 - w) + p[i1] * w

def augment_clip(p):
    Tn = p.shape[0]
    if np.random.rand() < 0.7:                                          # temporal change
        speed = np.random.uniform(0.8, 1.25); shift = np.random.uniform(-0.1, 0.1) * Tn
        grid = np.clip(np.arange(Tn) * speed + shift + Tn * (1 - speed) / 2, 0, Tn - 1)
        p = interp_time(p, grid)
    if np.random.rand() < 0.8:                                          # rotation
        th = np.deg2rad(np.random.uniform(-15, 15))
        R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]]); p = p @ R.T
    if np.random.rand() < 0.8: p = p * np.random.uniform(0.85, 1.15)    # scale
    if np.random.rand() < 0.8: p = p + np.random.uniform(-0.15, 0.15, (1, 1, 2))   # translation
    if np.random.rand() < 0.5: p = p + np.random.normal(0, 0.01, p.shape)
    if np.random.rand() < 0.3:                                          # occlusion
        j = np.random.randint(p.shape[1]); s = np.random.randint(0, Tn - 8); e = s + np.random.randint(4, 10)
        p = p.copy(); p[s:e, j] = np.nan; p = fill_missing(p)
    return p.astype(np.float32)

class ClipDS(Dataset):
    def __init__(s, P, y, mu, sd, augment=False):
        s.P, s.y, s.mu, s.sd, s.augment = P, np.asarray(y), mu, sd, augment
    def __len__(s): return len(s.P)
    def __getitem__(s, i):
        p = s.P[i]
        if s.augment: p = augment_clip(p)
        x = torch.from_numpy(make_features(p))
        d = torch.from_numpy(((kinematic_descriptors(p) - s.mu) / s.sd).astype(np.float32))
        return x, d, int(s.y[i])

def safe_split(items, labels, test_size, seed):
    try: return train_test_split(items, test_size=test_size, stratify=labels, random_state=seed)
    except ValueError: return train_test_split(items, test_size=test_size, random_state=seed)

def make_splits(W, holdout=None, seed=SEED, min_client=4):
    vids = np.unique(W["vid"])
    vlab = np.array([W["y"][W["vid"] == v][0] for v in vids])
    vinst = np.array([W["inst"][W["vid"] == v][0] for v in vids])
    if holdout is None:                           # video-level split -> no leakage between windows of one video
        trv, tev = safe_split(vids, vlab, 0.2, seed)
        l2 = vlab[np.isin(vids, trv)]
        trv, vav = safe_split(trv, l2, 0.125, seed)
        te_mask = np.isin(W["vid"], tev)
    else:                                         # leave-one-institution-out
        te_mask_v = vinst == holdout
        rest = vids[~te_mask_v]; trv, vav = safe_split(rest, vlab[~te_mask_v], 0.15, seed)
        te_mask = np.isin(W["vid"], vids[te_mask_v])
    tr_mask, va_mask = np.isin(W["vid"], trv), np.isin(W["vid"], vav)
    desc = np.stack([kinematic_descriptors(p) for p in W["P"][tr_mask]])
    mu, sd = desc.mean(0), desc.std(0) + 1e-6
    mk = lambda m, aug: ClipDS(W["P"][m], W["y"][m], mu, sd, aug)
    clients = []
    for k in np.unique(W["inst"][tr_mask]):
        m = tr_mask & (W["inst"] == k)
        if m.sum() >= min_client: clients.append(mk(m, True))
    return dict(train=mk(tr_mask, True), val=mk(va_mask, False), test=mk(te_mask, False), clients=clients,
                sizes=dict(train=int(tr_mask.sum()), val=int(va_mask.sum()), test=int(te_mask.sum()),
                           clients=[len(c) for c in clients]))

DATA = make_splits(W)
print(DATA["sizes"])

## 4. Proposed model: Hybrid Pooling-based Vision Transformer (PVT)
* **Conv stem** -> local spatio-temporal movement features, tokens on a (time x joint) grid
* **Pooled multi-head self-attention**: keys/values are average-pooled (reduces token redundancy and attention cost)
* **Token pooling** between stages (strided conv, halves the grid, widens channels), PiT/PVT style hierarchy
* **Skeleton branch**: MLP over kinematic descriptors, fused with the transformer feature
* Flags `use_pooling`, `use_transformer` drive the ablations

In [ ]:
class PooledMHSA(nn.Module):
    def __init__(s, dim, heads, pool):
        super().__init__()
        s.h, s.scale = heads, (dim // heads) ** -0.5
        s.q = nn.Linear(dim, dim); s.kv = nn.Linear(dim, dim * 2); s.proj = nn.Linear(dim, dim)
        s.pool = nn.AvgPool2d(pool, pool, ceil_mode=True) if pool > 1 else None
        s.norm = nn.LayerNorm(dim)
    def forward(s, x, hw):
        B, N, D = x.shape; H, Wd = hw
        q = s.q(x).reshape(B, N, s.h, D // s.h).transpose(1, 2)
        if s.pool is not None:
            xp = s.pool(x.transpose(1, 2).reshape(B, D, H, Wd)).flatten(2).transpose(1, 2)
        else: xp = x
        kv = s.kv(s.norm(xp)).reshape(B, -1, 2, s.h, D // s.h).permute(2, 0, 3, 1, 4)
        att = ((q @ kv[0].transpose(-2, -1)) * s.scale).softmax(-1)
        return s.proj((att @ kv[1]).transpose(1, 2).reshape(B, N, D))

class MLP(nn.Module):
    def __init__(s, d, r=2, drop=0.1):
        super().__init__(); s.net = nn.Sequential(nn.Linear(d, d * r), nn.GELU(), nn.Dropout(drop), nn.Linear(d * r, d), nn.Dropout(drop))
    def forward(s, x): return s.net(x)

class TransBlock(nn.Module):
    def __init__(s, d, heads, pool, drop=0.1):
        super().__init__()
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.attn, s.mlp, s.dp = PooledMHSA(d, heads, pool), MLP(d, 2, drop), nn.Dropout(drop)
    def forward(s, x, hw):
        x = x + s.dp(s.attn(s.n1(x), hw)); return x + s.mlp(s.n2(x))

class ConvBlock(nn.Module):               # replaces the transformer block in the "w/o Transformer" ablation
    def __init__(s, d, drop=0.1):
        super().__init__()
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.dw = nn.Conv2d(d, d, 3, padding=1, groups=d); s.pw = nn.Conv2d(d, d, 1)
        s.mlp = MLP(d, 2, drop)
    def forward(s, x, hw):
        B, N, D = x.shape; H, Wd = hw
        y = s.n1(x).transpose(1, 2).reshape(B, D, H, Wd)
        x = x + s.pw(F.gelu(s.dw(y))).flatten(2).transpose(1, 2)
        return x + s.mlp(s.n2(x))

class TokenPool(nn.Module):               # token pooling between stages
    def __init__(s, din, dout, down):
        super().__init__(); s.down = down
        s.conv = nn.Conv2d(din, dout, 3, stride=2, padding=1) if down else nn.Conv2d(din, dout, 1)
        s.norm = nn.LayerNorm(dout)
    def forward(s, x, hw):
        B, N, D = x.shape; H, Wd = hw
        y = s.conv(x.transpose(1, 2).reshape(B, D, H, Wd))
        h2, w2 = y.shape[-2:]
        return s.norm(y.flatten(2).transpose(1, 2)), (h2, w2)

class HybridPVT(nn.Module):
    def __init__(s, in_ch=6, Tn=T, Jn=J, desc_dim=None, dims=(48, 96, 128), depths=(1, 1, 2), heads=(2, 4, 4),
                 use_transformer=True, use_pooling=True, use_skeleton=True, n_classes=2, drop=0.1):
        super().__init__()
        desc_dim = desc_dim or DESC_DIM
        s.use_skeleton = use_skeleton
        s.stem = nn.Sequential(nn.Conv2d(in_ch, 32, 3, padding=1), nn.BatchNorm2d(32), nn.GELU(),
                               nn.Conv2d(32, dims[0], 3, stride=2, padding=1), nn.BatchNorm2d(dims[0]), nn.GELU())
        h0, w0 = math.ceil(Tn / 2), math.ceil(Jn / 2); s.hw0 = (h0, w0)
        s.pos = nn.Parameter(torch.zeros(1, h0 * w0, dims[0])); nn.init.trunc_normal_(s.pos, std=0.02)
        pool_ratio = 2 if use_pooling else 1
        s.stages, s.trans = nn.ModuleList(), nn.ModuleList()
        for i, (d, n, hd) in enumerate(zip(dims, depths, heads)):
            if i > 0: s.trans.append(TokenPool(dims[i - 1], d, down=use_pooling))
            blocks = [TransBlock(d, hd, pool_ratio, drop) if use_transformer else ConvBlock(d, drop) for _ in range(n)]
            s.stages.append(nn.ModuleList(blocks))
        s.norm = nn.LayerNorm(dims[-1])
        s.skel = nn.Sequential(nn.Linear(desc_dim, 64), nn.GELU(), nn.Dropout(drop), nn.Linear(64, 64), nn.GELU()) if use_skeleton else None
        s.head = nn.Sequential(nn.Dropout(drop), nn.Linear(dims[-1] + (64 if use_skeleton else 0), n_classes))
    def forward(s, x, d=None):
        f = s.stem(x); B, C, H, Wd = f.shape
        t = f.flatten(2).transpose(1, 2) + s.pos; hw = (H, Wd)
        for i, blocks in enumerate(s.stages):
            if i > 0: t, hw = s.trans[i - 1](t, hw)
            for b in blocks: t = b(t, hw)
        z = s.norm(t).mean(1)
        if s.use_skeleton: z = torch.cat([z, s.skel(d)], 1)
        return s.head(z)

def count_params(m): return sum(p.numel() for p in m.parameters())
_m = HybridPVT(); _x = torch.randn(4, 6, T, J); _d = torch.randn(4, DESC_DIM)
print("PVT grid:", _m.hw0, "| out:", _m(_x, _d).shape, "| params:", count_params(_m))

## 5. Baselines: GCN-GRU, DCNN, Bi-GRU-LSTM, TransCP-Net
All take the same `(B, 6, T, J)` input (descriptor input is ignored).

In [ ]:
def _adj():
    A = np.eye(J)
    for j, p in enumerate(PARENT): A[j, p] = A[p, j] = 1
    A = A / A.sum(1, keepdims=True)
    return torch.tensor(A, dtype=torch.float32)

class GCNGRU(nn.Module):
    def __init__(s, in_ch=6, h=64, n_classes=2, drop=0.2):
        super().__init__(); s.register_buffer("A", _adj())
        s.g1, s.g2 = nn.Linear(in_ch, h), nn.Linear(h, h)
        s.gru = nn.GRU(h, h, batch_first=True); s.drop = nn.Dropout(drop); s.fc = nn.Linear(h, n_classes)
    def forward(s, x, d=None):
        z = x.permute(0, 2, 3, 1)                                   # (B,T,J,C)
        z = F.relu(torch.einsum("jk,btkc->btjc", s.A, s.g1(z)))
        z = F.relu(torch.einsum("jk,btkc->btjc", s.A, s.g2(z))).mean(2)
        _, h = s.gru(z); return s.fc(s.drop(h[-1]))

class DCNN(nn.Module):
    def __init__(s, in_ch=6, n_classes=2, drop=0.3):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        s.f = nn.Sequential(blk(in_ch, 32), blk(32, 64), blk(64, 128), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        s.fc = nn.Sequential(nn.Dropout(drop), nn.Linear(128, n_classes))
    def forward(s, x, d=None): return s.fc(s.f(x))

class BiGRULSTM(nn.Module):
    def __init__(s, in_ch=6, h=64, n_classes=2, drop=0.3):
        super().__init__()
        s.inp = nn.Linear(in_ch * J, 128); s.bigru = nn.GRU(128, h, batch_first=True, bidirectional=True)
        s.lstm = nn.LSTM(2 * h, h, batch_first=True); s.drop = nn.Dropout(drop); s.fc = nn.Linear(h, n_classes)
    def forward(s, x, d=None):
        z = x.permute(0, 2, 1, 3).flatten(2)                        # (B,T,C*J)
        z, _ = s.bigru(F.relu(s.inp(z))); _, (h, _) = s.lstm(z)
        return s.fc(s.drop(h[-1]))

class TransCPNet(nn.Module):             # temporal transformer encoder over per-frame skeleton tokens
    def __init__(s, in_ch=6, dim=96, layers=2, heads=4, n_classes=2, drop=0.1):
        super().__init__()
        s.emb = nn.Linear(in_ch * J, dim); s.cls = nn.Parameter(torch.zeros(1, 1, dim))
        s.pos = nn.Parameter(torch.zeros(1, T + 1, dim)); nn.init.trunc_normal_(s.pos, std=0.02)
        el = nn.TransformerEncoderLayer(dim, heads, dim * 2, drop, batch_first=True, norm_first=True, activation="gelu")
        s.enc = nn.TransformerEncoder(el, layers, enable_nested_tensor=False)
        s.norm = nn.LayerNorm(dim); s.fc = nn.Linear(dim, n_classes)
    def forward(s, x, d=None):
        z = s.emb(x.permute(0, 2, 1, 3).flatten(2))
        z = torch.cat([s.cls.expand(z.size(0), -1, -1), z], 1) + s.pos
        return s.fc(s.norm(s.enc(z)[:, 0]))

for _n, _c in [("GCN-GRU", GCNGRU), ("DCNN", DCNN), ("Bi-GRU-LSTM", BiGRULSTM), ("TransCP-Net", TransCPNet)]:
    print(f"{_n:12s} params={count_params(_c()):,} out={tuple(_c()(_x).shape)}")

## 6. Training: local training, FedAvg (+FedProx / DP option), centralised training, metrics, latency
Communication cost per round = 2 x clients x parameters x 4 bytes (upload + download, fp32).

In [ ]:
def make_loader(ds, train, bs=BATCH):
    if train:
        y = np.asarray(ds.y); cnt = np.bincount(y, minlength=2).astype(float); cnt[cnt == 0] = 1
        w = (1.0 / cnt)[y]
        sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=len(ds), replacement=True)
        return DataLoader(ds, batch_size=bs, sampler=sampler, drop_last=len(ds) > bs)
    return DataLoader(ds, batch_size=64, shuffle=False)

def local_train(model, loader, epochs, lr, mu=0.0, opt=None):
    model.train()
    if opt is None: opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    gp = [p.detach().clone() for p in model.parameters()] if mu > 0 else None
    for _ in range(epochs):
        for x, d, y in loader:
            x, d, y = x.to(DEVICE), d.to(DEVICE), y.to(DEVICE)
            if x.size(0) < 2: continue
            opt.zero_grad()
            loss = F.cross_entropy(model(x, d), y, label_smoothing=0.05)
            if mu > 0: loss = loss + mu / 2 * sum(((p - g) ** 2).sum() for p, g in zip(model.parameters(), gp))
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()

@torch.no_grad()
def predict(model, ds):
    model.eval(); P, Y = [], []
    for x, d, y in make_loader(ds, False):
        P.append(F.softmax(model(x.to(DEVICE), d.to(DEVICE)), -1)[:, 1].cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

def safe_auc(y, p): return roc_auc_score(y, p) if len(np.unique(y)) > 1 else float("nan")

def best_threshold(y, p):                         # Youden-J on validation
    if len(np.unique(y)) < 2: return 0.5
    fpr, tpr, th = roc_curve(y, p); th = th[np.argmax(tpr - fpr)]
    return float(th) if np.isfinite(th) else 0.5

def compute_metrics(y, p, thr=0.5):
    yh = (p >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    return dict(Accuracy=accuracy_score(y, yh), Sensitivity=tp / max(tp + fn, 1), Specificity=tn / max(tn + fp, 1),
                Precision=precision_score(y, yh, zero_division=0), F1=f1_score(y, yh, zero_division=0), AUC=safe_auc(y, p))

def aggregate(gstate, locals_, sizes, dp_clip=0.0, dp_sigma=0.0):
    w = np.array(sizes, float); w /= w.sum(); n = len(locals_)
    scales = np.ones(n)
    if dp_clip > 0:
        for i, ls in enumerate(locals_):
            nrm = math.sqrt(sum(((ls[k] - g).float() ** 2).sum().item() for k, g in gstate.items() if torch.is_floating_point(g)))
            scales[i] = min(1.0, dp_clip / (nrm + 1e-12))
    new = {}
    for k, g in gstate.items():
        if not torch.is_floating_point(g): new[k] = locals_[0][k].clone(); continue
        delta = sum(w[i] * scales[i] * (locals_[i][k] - g) for i in range(n))
        if dp_sigma > 0 and dp_clip > 0: delta = delta + torch.randn_like(g) * dp_sigma * dp_clip / n
        new[k] = g + delta
    return new

def federated_train(model_fn, clients, val_ds, rounds, local_epochs, lr, mu=0.0, dp_clip=0.0, dp_sigma=0.0, patience=PATIENCE, verbose=True):
    gm = model_fn(); n_params = count_params(gm); hist = []; best_auc, best_state, bad = -1, None, 0
    for r in range(1, rounds + 1):
        gstate = {k: v.detach().cpu().clone() for k, v in gm.state_dict().items()}
        locals_, sizes = [], []
        for ds in clients:
            lm = model_fn(); lm.load_state_dict(gstate)
            local_train(lm, make_loader(ds, True), local_epochs, lr, mu)
            locals_.append({k: v.detach().cpu().clone() for k, v in lm.state_dict().items()}); sizes.append(len(ds))
        gm.load_state_dict(aggregate(gstate, locals_, sizes, dp_clip, dp_sigma))
        pv, yv = predict(gm, val_ds); auc = safe_auc(yv, pv); acc = accuracy_score(yv, pv >= 0.5)
        hist.append(dict(round=r, val_auc=auc, val_acc=acc))
        if verbose and (r % 2 == 0 or r == 1): print(f"  round {r:3d} | val AUC {auc:.3f} | val acc {acc:.3f}")
        score = auc if np.isfinite(auc) else acc
        if score > best_auc + 1e-4: best_auc, best_state, bad = score, copy.deepcopy(gm.state_dict()), 0
        else: bad += 1
        if bad >= patience: break
    gm.load_state_dict(best_state)
    aucs = np.array([h["val_auc"] for h in hist]); aucs = np.nan_to_num(aucs, nan=0.0)
    tgt = 0.95 * aucs.max(); r_tgt = int(np.argmax(aucs >= tgt)) + 1
    mb = 2 * len(clients) * n_params * 4 / 1e6
    return gm, hist, dict(rounds_run=len(hist), rounds_to_95pct_best=r_tgt, comm_mb_per_round=mb, total_comm_mb=mb * len(hist), n_clients=len(clients))

def central_train(model_fn, train_ds, val_ds, epochs, lr, patience=PATIENCE * 2, verbose=True):
    model = model_fn(); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    loader = make_loader(train_ds, True); hist = []; best_auc, best_state, bad = -1, None, 0
    for e in range(1, epochs + 1):
        local_train(model, loader, 1, lr, opt=opt)
        pv, yv = predict(model, val_ds); auc = safe_auc(yv, pv); acc = accuracy_score(yv, pv >= 0.5)
        hist.append(dict(round=e, val_auc=auc, val_acc=acc))
        if verbose and (e % 4 == 0 or e == 1): print(f"  epoch {e:3d} | val AUC {auc:.3f} | val acc {acc:.3f}")
        score = auc if np.isfinite(auc) else acc
        if score > best_auc + 1e-4: best_auc, best_state, bad = score, copy.deepcopy(model.state_dict()), 0
        else: bad += 1
        if bad >= patience: break
    model.load_state_dict(best_state)
    return model, hist, dict(rounds_run=0, rounds_to_95pct_best=np.nan, comm_mb_per_round=0.0, total_comm_mb=0.0, n_clients=1)

@torch.no_grad()
def measure_latency(model, n=50):
    model.eval(); x = torch.randn(1, 6, T, J, device=DEVICE); d = torch.randn(1, DESC_DIM, device=DEVICE)
    for _ in range(5): model(x, d)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n): model(x, d)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000

def run_experiment(name, builder, mode, data, rounds=ROUNDS, verbose=True):
    print(f"\n=== {name} [{mode}] ===")
    set_seed(); mf = lambda: builder().to(DEVICE)
    if mode == "fed":
        model, hist, info = federated_train(mf, data["clients"], data["val"], rounds, LOCAL_EPOCHS, LR, FEDPROX_MU, DP_CLIP, DP_SIGMA, verbose=verbose)
    else:
        model, hist, info = central_train(mf, data["train"], data["val"], rounds * LOCAL_EPOCHS, LR, verbose=verbose)
    pv, yv = predict(model, data["val"]); thr = best_threshold(yv, pv)
    pt, yt = predict(model, data["test"]); m = compute_metrics(yt, pt, thr)
    res = dict(Model=name, Mode=mode, **m, Threshold=thr, Latency_ms=measure_latency(model), Params=count_params(model),
               Rounds_run=info["rounds_run"], Rounds_to_95pct=info["rounds_to_95pct_best"], Comm_MB_total=info["total_comm_mb"])
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in res.items() if k not in ("Model", "Mode")})
    return dict(row=res, hist=hist, probs=pt, y=yt, model=model, thr=thr)

## 7. Experiments: ablation study (Full / w/o Pooling / w/o FL / w/o Transformer)

In [ ]:
ABL = {
    "Full Model (PVT + Pooling + FL)": (lambda: HybridPVT(), "fed"),
    "w/o Pooling Mechanism (FL)":      (lambda: HybridPVT(use_pooling=False), "fed"),
    "w/o Federated Learning (central)":(lambda: HybridPVT(), "central"),
    "w/o Transformer (FL)":            (lambda: HybridPVT(use_transformer=False), "fed"),
}
RESULTS = {}
for name, (b, mode) in ABL.items():
    RESULTS[name] = run_experiment(name, b, mode, DATA)
ablation_df = pd.DataFrame([RESULTS[n]["row"] for n in ABL]).set_index("Model")
cols = ["Accuracy", "Sensitivity", "Specificity", "Precision", "F1", "AUC", "Latency_ms", "Params", "Rounds_to_95pct", "Comm_MB_total"]
display(ablation_df[cols].round(4)) if "display" in globals() else print(ablation_df[cols].round(4))
ablation_df.to_csv(f"{OUT}/ablation_results.csv")

## 8. Comparison with baseline models
Baselines default to centralised training (set `BASELINE_MODE = "fed"` in the config to train them with the same FL protocol).

In [ ]:
BASE = {"GCN-GRU": GCNGRU, "DCNN": DCNN, "Bi-GRU-LSTM": BiGRULSTM, "TransCP-Net": TransCPNet}
for name, cls in BASE.items():
    RESULTS[name] = run_experiment(name, lambda c=cls: c(), BASELINE_MODE, DATA)
RESULTS["Proposed Hybrid PVT-FL"] = RESULTS["Full Model (PVT + Pooling + FL)"]
order = ["GCN-GRU", "DCNN", "Bi-GRU-LSTM", "TransCP-Net", "Proposed Hybrid PVT-FL"]
comp_df = pd.DataFrame([{**RESULTS[n]["row"], "Model": n} for n in order]).set_index("Model")
print(comp_df[cols + ["Mode"]].round(4))
comp_df.to_csv(f"{OUT}/comparison_results.csv")

## 9. Institution-independence: leave-one-institution-out (LOIO) with FL
Each institution is held out in turn; the model is trained federatedly on the remaining ones and tested on the unseen institution.

In [ ]:
loio_rows = []
for k in np.unique(W["inst"]):
    sp = make_splits(W, holdout=int(k))
    if sp["sizes"]["test"] < 5 or len(sp["clients"]) < 2: continue
    r = run_experiment(f"LOIO hold-out inst {k}", lambda: HybridPVT(), "fed", sp, rounds=LOIO_ROUNDS, verbose=False)
    loio_rows.append(dict(Held_out_institution=int(k), N_test=sp["sizes"]["test"], **{m: r["row"][m] for m in ["Accuracy", "Sensitivity", "Specificity", "F1", "AUC"]}))
loio_df = pd.DataFrame(loio_rows)
print(loio_df.round(4)); print("\nmean +- std\n", loio_df.drop(columns=["Held_out_institution"]).agg(["mean", "std"]).round(4))
loio_df.to_csv(f"{OUT}/loio_results.csv", index=False)

## 10. Figures: ROC curves, metric bars, FL convergence, communication efficiency, confusion matrix

In [ ]:
def plot_roc(names, title, fn):
    plt.figure(figsize=(6, 5))
    for n in names:
        r = RESULTS[n]
        if len(np.unique(r["y"])) > 1:
            fpr, tpr, _ = roc_curve(r["y"], r["probs"]); plt.plot(fpr, tpr, label=f"{n} (AUC={r['row']['AUC']:.3f})")
    plt.plot([0, 1], [0, 1], "k--", lw=.8); plt.xlabel("1 - Specificity"); plt.ylabel("Sensitivity"); plt.title(title)
    plt.legend(fontsize=7, loc="lower right"); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig(f"{OUT}/{fn}", dpi=200); plt.show()

plot_roc(order, "ROC: proposed vs baselines", "roc_comparison.png")
plot_roc(list(ABL), "ROC: ablation", "roc_ablation.png")

def bar_metrics(df, title, fn):
    ms = ["Accuracy", "Sensitivity", "Specificity", "Precision", "F1", "AUC"]
    ax = df[ms].plot(kind="bar", figsize=(11, 4.5), width=.8); ax.set_ylim(0, 1.05); ax.set_title(title); ax.grid(axis="y", alpha=.3)
    plt.xticks(rotation=20, ha="right"); plt.legend(ncol=6, fontsize=8, loc="lower center"); plt.tight_layout(); plt.savefig(f"{OUT}/{fn}", dpi=200); plt.show()
bar_metrics(comp_df, "Model comparison (test set)", "bars_comparison.png")
bar_metrics(ablation_df, "Ablation study (test set)", "bars_ablation.png")

fed_names = [n for n in ABL if ABL[n][1] == "fed"]
plt.figure(figsize=(6.5, 4.5))
for n in fed_names:
    h = pd.DataFrame(RESULTS[n]["hist"]); plt.plot(h["round"], h["val_auc"], marker="o", ms=3, label=n)
plt.xlabel("Communication round"); plt.ylabel("Validation AUC"); plt.title("FL convergence"); plt.legend(fontsize=7); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT}/fl_convergence.png", dpi=200); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ablation_df[["Latency_ms"]].plot(kind="bar", ax=ax[0], legend=False, color="tab:green"); ax[0].set_title("Inference latency (ms / clip)"); ax[0].tick_params(axis="x", rotation=25)
ablation_df[["Comm_MB_total"]].plot(kind="bar", ax=ax[1], legend=False, color="tab:orange"); ax[1].set_title("Total communication (MB)"); ax[1].tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.savefig(f"{OUT}/efficiency.png", dpi=200); plt.show()

full = RESULTS["Full Model (PVT + Pooling + FL)"]
cm = confusion_matrix(full["y"], (full["probs"] >= full["thr"]).astype(int), labels=[0, 1])
plt.figure(figsize=(3.8, 3.4)); plt.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2): plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=13)
plt.xticks([0, 1], ["Typical", "CP risk"]); plt.yticks([0, 1], ["Typical", "CP risk"]); plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix (proposed)")
plt.tight_layout(); plt.savefig(f"{OUT}/confusion_matrix.png", dpi=200); plt.show()

## 11. Save model and final summary

In [ ]:
torch.save(RESULTS["Full Model (PVT + Pooling + FL)"]["model"].state_dict(), f"{OUT}/hybrid_pvt_fl.pt")
print("DATA SOURCE:", DATA_SOURCE)
print("\nCOMPARISON\n", comp_df[["Accuracy", "Sensitivity", "Specificity", "Precision", "F1", "AUC", "Latency_ms", "Comm_MB_total"]].round(4).to_string())
print("\nABLATION\n", ablation_df[["Accuracy", "Sensitivity", "Specificity", "Precision", "F1", "AUC", "Rounds_to_95pct"]].round(4).to_string())
print("\nFiles in", OUT, ":", sorted(os.listdir(OUT)))

### Notes and limitations
* **Run on real data:** if the repository stores keypoints / videos with names that do not reveal the class, edit `label_from_path` (Section 2) or load a labels CSV there. Install `opencv-python` and `mediapipe` for video input.
* **Leakage control:** splits are made at *video* level; if the dataset provides infant ids, split on infant id instead (replace `W["vid"]`).
* **Institutions:** FL clients are simulated (real site ids if the data has them, otherwise Dirichlet label-skew pseudo-sites). FL here is simulated in one process; communication is counted analytically.
* **Thresholds:** the operating point is chosen on the validation set (Youden-J). For screening, raise sensitivity by lowering the threshold.
* **Statistics:** single seed by default; for the paper, repeat with several seeds / k-fold and report mean +- std and confidence intervals.
* **Clinical claim:** results are research-grade only; prospective validation on a sufficiently large infant cohort is required.